# 09: Personalization with Graph-Powered RAG

This notebook demonstrates how to create personalized retrieval and content generation using graph patterns and entity relationships.

## Overview

We'll:
1. Find notes based on entity co-occurrence patterns
2. Create personalized retrieval based on user context
3. Build retrieval chains that prioritize related content
4. Generate personalized summaries using graph context


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain.graphs import Neo4jGraph
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnableLambda

print("✅ Additional libraries imported")


In [ ]:
# Create ChatOpenAI instance using LiteLLM proxy
proxy_base_url = f"http://{settings.litellm_proxy_host}:{settings.litellm_proxy_port}"

llm = ChatOpenAI(
    model=settings.litellm_proxy_model or "lm_studio/qwen3-coder-30b",
    base_url=f"{proxy_base_url}/v1",
    api_key=settings.openai_api_key or "not-needed",
    temperature=0,
)

# Create custom embeddings class for LiteLLM proxy
class ProxyEmbeddings(OpenAIEmbeddings):
    """Custom embeddings class that uses LiteLLM proxy."""
    def __init__(self, proxy_base_url: str, api_key: str, model: str, **kwargs):
        api_url = f"{proxy_base_url}/v1"
        super().__init__(
            openai_api_base=api_url,
            openai_api_key=api_key,
            model=model,
            **kwargs
        )

embedding_model = ProxyEmbeddings(
    proxy_base_url=proxy_base_url,
    api_key=settings.openai_api_key or "",
    model=settings.litellm_proxy_embedding_model,
)

# Create Neo4jGraph for graph queries
kg = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database
)

print("✅ LangChain components configured")


## Personalized Retrieval Based on Entity Context

Create a retrieval function that finds notes based on entity relationships and co-occurrence.


In [ ]:
def get_personalized_notes(entity_name: str, max_notes: int = 10):
    """Get notes personalized to a specific entity based on graph relationships."""
    query = """
    MATCH (e:Entity {name: $entity_name})<-[:CONTAINS]-(source:Note)
    
    // Find other entities that co-occur with this entity
    OPTIONAL MATCH (source)-[:CONTAINS]->(related_entity:Entity)
    WHERE related_entity <> e
    
    // Find notes that share related entities
    OPTIONAL MATCH (related_entity)<-[:CONTAINS]-(related_note:Note)
    WHERE related_note <> source
    
    WITH source, related_note, count(DISTINCT related_entity) AS shared_entities
    ORDER BY shared_entities DESC
    
    RETURN DISTINCT 
        coalesce(related_note.file_path, source.file_path) AS file_path,
        coalesce(related_note.text, source.text) AS text,
        coalesce(related_note.content, source.content) AS content,
        shared_entities AS relevance_score
    LIMIT $max_notes
    """
    
    results = kg.query(query, params={"entity_name": entity_name, "max_notes": max_notes})
    return results

# Example: Get personalized notes for an entity
entity_name = "Project"  # Change this to an entity from your graph

personalized_notes = get_personalized_notes(entity_name, max_notes=5)

if personalized_notes:
    print(f"Personalized notes for entity '{entity_name}':\n")
    for i, note in enumerate(personalized_notes, 1):
        print(f"{i}. {note.get('file_path', 'N/A')}")
        print(f"   Relevance: {note.get('relevance_score', 0)} shared entities")
        print(f"   Preview: {note.get('text', '')[:100]}...\n")
else:
    print(f"No personalized notes found for '{entity_name}'")
    print("Try a different entity name from your graph")


## Personalized Vector Search with Graph Context

Create a vector search retriever that uses graph patterns to personalize results.


In [ ]:
# Create personalized vector search with graph context
# This retrieval query boosts notes that share entities with the search results
kg_personalized_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database,
    index_name=settings.neo4j_vector_index_name,
    retrieval_query="""
    WITH node AS note, score AS searchScore
    
    // Find entities in this note
    OPTIONAL MATCH (note)-[:CONTAINS]->(e:Entity)
    
    // Find other notes that share entities (personalization boost)
    OPTIONAL MATCH (note)-[:CONTAINS]->(e:Entity)<-[:CONTAINS]-(related:Note)
    WHERE related <> note
    
    // Count shared entities for personalization score
    WITH note, searchScore, count(DISTINCT related) AS relatedNoteCount
    
    // Boost score based on entity relationships
    RETURN note.text AS text,
           (searchScore * (1.0 + relatedNoteCount * 0.1)) AS personalized_score,
           relatedNoteCount,
           {file_path: note.file_path, 
            file_name: note.file_name,
            related_notes: relatedNoteCount} AS metadata
    ORDER BY personalized_score DESC
    LIMIT 10
    """
)

print("✅ Personalized vector search configured")


In [ ]:
# Test personalized search
search_query = "project ideas and development"

results = kg_personalized_search.similarity_search(search_query, k=5)

print(f"Personalized search results for: '{search_query}'\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:150]}...")
    if hasattr(doc, 'metadata') and doc.metadata:
        print(f"   Related notes: {doc.metadata.get('related_notes', 0)}")
        print(f"   File: {doc.metadata.get('file_path', 'N/A')}\n")


## Personalized Content Generation

Create a chain that generates personalized summaries using graph context.


In [ ]:
# Create personalized retrieval function
def personalized_retriever(query: str, entity_context: str = None, k: int = 5):
    """Retrieve notes with personalization based on entity context."""
    # First, do vector search
    vector_results = kg_personalized_search.similarity_search(query, k=k)
    
    # If entity context provided, boost notes related to that entity
    if entity_context:
        entity_notes = get_personalized_notes(entity_context, max_notes=k)
        # Combine and deduplicate
        seen_paths = set()
        combined = []
        
        # Add entity-related notes first (higher priority)
        for note in entity_notes:
            path = note.get('file_path')
            if path and path not in seen_paths:
                combined.append(note.get('text', ''))
                seen_paths.add(path)
        
        # Add vector search results
        for doc in vector_results:
            path = doc.metadata.get('file_path') if hasattr(doc, 'metadata') else None
            if path and path not in seen_paths:
                combined.append(doc.page_content)
                seen_paths.add(path)
        
        return "\n\n".join(combined[:k])
    
    return "\n\n".join([doc.page_content for doc in vector_results])

# Create prompt template for personalized generation
personalization_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that creates personalized summaries based on user context.
Use the provided context to create a relevant, personalized summary that highlights connections
and relationships between different pieces of information."""),
    ("human", """Based on the following context, create a personalized summary for someone interested in: {user_interest}

Context:
{context}

Create a concise, personalized summary that highlights the most relevant information and connections.""")
])

# Create personalized generation chain
personalized_chain = (
    {
        "context": RunnableLambda(lambda x: personalized_retriever(x["query"], x.get("entity_context"))),
        "user_interest": lambda x: x["query"]
    }
    | personalization_prompt
    | llm
    | StrOutputParser()
)

print("✅ Personalized content generation chain created")


In [ ]:
# Test personalized generation
result = personalized_chain.invoke({
    "query": "project ideas and development tasks",
    "entity_context": "Project"  # Optional: provide entity for additional personalization
})

print("Personalized Summary:")
print(result)
